# SIAT-LLMD Healthy Gait Anomaly Detection — sEMG Pipeline
**Pipeline:** Clone repo → install deps → load SIAT-LLMD dataset → filter + normalize sEMG → sliding window → train LSTM & Transformer autoencoders → compute thresholds → inject synthetic anomalies → score test subjects → evaluate (Recall / Precision / F1) → visualise.

## ── Environment setup ────────────────────────────────────────────────────────
Clone/update the repo, then install extra dependencies.

In [ ]:
!git clone https://github.com/GM-10/healthy-gait-anomaly-detection.git 2>/dev/null \
    || (cd healthy-gait-anomaly-detection && git pull origin main)
%cd healthy-gait-anomaly-detection
!pip install -q scipy pmdarima scikit-learn pyyaml matplotlib pandas numpy torch

## Configuration
Edit BASE_DIR, MOVEMENTS, MODELS, and SEVERITIES here. All subsequent cells read from these variables.

In [ ]:
# ── Paths & experiment settings ──────────────────────────────────────────────
import os, sys, json, math, pickle, warnings, logging, time
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

BASE_DIR   = '/kaggle/input/datasets/preritashukla/siat-llmd/SIAT_LLMD20230404'
OUT_ROOT   = '/kaggle/working/outputs'
MOVEMENTS  = ['WAK', 'UPS', 'SITDN']
MODELS     = ['lstm', 'transformer']        # SARIMA excluded (too slow on Kaggle)
SEVERITIES = ['mild', 'moderate', 'severe']

USE_SEMI_SUPERVISED = True
AUGMENTATION_FRACTION = 0.2
LAMBDA_CLS = 0.5

os.makedirs(OUT_ROOT, exist_ok=True)

# Make sure the repo root is importable
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device           : {DEVICE}')
print(f'BASE_DIR exists  : {os.path.isdir(BASE_DIR)}')
print(f'Movements        : {MOVEMENTS}')
print(f'Models           : {MODELS}')
print(f'Severities       : {SEVERITIES}')

## Imports
Imports come directly from the cloned repo modules.

In [ ]:
# ── Import pipeline modules from the cloned repo ─────────────────────────────
from semg_pipeline.loader import (
    load_semg_trial, build_trial_paths,
    SEMG_CHANNELS, TRAIN_SUBS, VAL_SUBS, TEST_SUBS,
)
from semg_pipeline.filter     import apply_semg_filter_chain
from semg_pipeline.normalizer import fit_scaler, apply_scaler, save_scaler, load_scaler
from semg_pipeline.windower   import create_semg_windows
from semg_pipeline.anomaly_scorer import (
    compute_threshold, label_windows, build_output_rows, save_train_errors,
)
from semg_pipeline.models.lstm_model        import LSTMModel
from semg_pipeline.models.transformer_model import TransformerModel
from utils.synthetic_anomalies import (
    inject_amplitude_scale, inject_time_warp,
    inject_time_shift, inject_combined, DEFAULT_SEVERITIES,
)

WINDOW_SIZE  = 1920    # 1 second at 1920 Hz
OVERLAP_SIZE = 960     # 50 % overlap
FS           = 1920.0
MODEL_DISPLAY = {'lstm': 'LSTM', 'transformer': 'Transformer', 'sarima': 'SARIMA'}

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s')
logger = logging.getLogger('semg_nb')

print(f'sEMG channels ({len(SEMG_CHANNELS)}): {SEMG_CHANNELS}')
print('Train subjects :', TRAIN_SUBS[:3], '…', TRAIN_SUBS[-1])
print('Test  subjects :', TEST_SUBS)
print('All imports OK')

## Pipeline Helpers
Utility functions for loading, filtering, windowing, and building anomaly injection conditions.

In [ ]:
# -- Pipeline helper functions

def process_trial(base_dir, subject, movement,
                  scaling_params=None, filter_only=False):
    # Load, filter, scale, and window one trial. Returns (df, windows, meta).
    data_path, label_path = build_trial_paths(base_dir, subject, movement)
    if not os.path.exists(data_path) or not os.path.exists(label_path):
        return None, None, None
    try:
        df = load_semg_trial(data_path, label_path, active_only=True)
    except Exception as exc:
        logger.warning(f'  [load] {subject}/{movement}: {exc}')
        return None, None, None
    if df.empty:
        return None, None, None

    df = apply_semg_filter_chain(df, fs=FS)
    if filter_only:
        return df, None, None
    if scaling_params is not None:
        df = apply_scaler(df, scaling_params)
    windows, meta = create_semg_windows(df, WINDOW_SIZE, OVERLAP_SIZE)
    if len(windows) == 0:
        return df, None, None
    return df, windows, meta


def collect_split_windows(subjects, movements, scaling_params, split_label='train'):
    # Collect and concatenate windows for a full subject split.
    all_windows, all_meta = [], []
    n, done = len(subjects) * len(movements), 0
    for sub in subjects:
        for mov in movements:
            done += 1
            print(f'\r  [{split_label}] {sub}/{mov}  ({done}/{n})', end='', flush=True)
            _, windows, meta = process_trial(BASE_DIR, sub, mov, scaling_params)
            if windows is not None:
                all_windows.append(windows)
                all_meta.extend(meta)
    print()
    if not all_windows:
        return np.empty((0, WINDOW_SIZE, len(SEMG_CHANNELS)), dtype=np.float32), []
    return np.concatenate(all_windows, axis=0), all_meta


def build_anomaly_conditions(severity_levels):
    # Return (inject_fn, kwargs, anomaly_type, severity_label) tuples.
    inject_fns = [
        (inject_amplitude_scale, 'amplitude_scale'),
        (inject_time_warp,       'time_warp'),
        (inject_time_shift,      'time_shift'),
    ]
    conditions = []
    for fn, atype in inject_fns:
        for level in severity_levels:
            conditions.append((fn, {'severity': DEFAULT_SEVERITIES[level]}, atype, level))
    conditions.append((inject_combined, {}, 'combined', 'moderate'))
    return conditions

print('Helper functions defined')

## Phase 1 — Min-Max Scaler
Fit on training subjects only (Sub01–Sub30). Cached to `scaler_params.json` so re-runs are instant.

In [ ]:
# ── Phase 1: Fit Min-Max scaler on TRAINING subjects only ────────────────────
SEMG_OUT  = os.path.join(OUT_ROOT, 'sEMG')
MODEL_DIR = os.path.join(SEMG_OUT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

scaler_path = os.path.join(SEMG_OUT, 'scaler_params.json')

if os.path.exists(scaler_path):
    print(f'Loading existing scaler from {scaler_path}')
    scaling_params = load_scaler(scaler_path)
else:
    print('Fitting scaler on raw filtered train data …')
    raw_dfs = []
    for sub in TRAIN_SUBS:
        for mov in MOVEMENTS:
            df, _, _ = process_trial(BASE_DIR, sub, mov, scaling_params=None, filter_only=True)
            if df is not None and not df.empty:
                raw_dfs.append(df)
    print(f'  Collected {len(raw_dfs)} train trial DataFrames')
    scaling_params = fit_scaler(raw_dfs)
    save_scaler(scaling_params, scaler_path)
    print(f'Scaler fitted and saved -> {scaler_path}')

print(f'Scaler covers {len(scaling_params)} channels')

## Phase 2 — Sliding Window Collection
Segments each trial into 1-second windows (1920 samples, 50% overlap). Only cycle-complete windows (same Group label) are kept.

In [ ]:
# ── Phase 2: Collect sliding windows for train and validation splits ─────────
print('Collecting TRAIN windows …')
train_windows, train_meta = collect_split_windows(
    TRAIN_SUBS, MOVEMENTS, scaling_params, 'train'
)
print(f'  Train windows shape : {train_windows.shape}')

print('Collecting VALIDATION windows …')
val_windows, val_meta = collect_split_windows(
    VAL_SUBS, MOVEMENTS, scaling_params, 'val'
)
print(f'  Val windows shape   : {val_windows.shape}')

## Phase 3a — LSTM Autoencoder Training
One LSTM autoencoder trained independently per sEMG channel (9 total). Weights cached to disk.

In [ ]:
# ── Phase 3a: Train LSTM autoencoder per sEMG channel ────────────────────────
os.makedirs(os.path.join(MODEL_DIR, 'lstm'), exist_ok=True)

lstm_models = []
for ch_idx, ch_name in enumerate(SEMG_CHANNELS):
    save_path = os.path.join(MODEL_DIR, 'lstm', f'lstm_ch{ch_idx:02d}.pt')
    m = LSTMModel(channel_name=ch_name, window_size=WINDOW_SIZE)
    if os.path.exists(save_path):
        print(f'  [LSTM] ch {ch_idx+1:02d}/{len(SEMG_CHANNELS)}  {ch_name:<30s} — loaded')
        m.load(save_path)
    else:
        print(f'  [LSTM] ch {ch_idx+1:02d}/{len(SEMG_CHANNELS)}  {ch_name:<30s} — training')
        train_ch = train_windows[:, :, ch_idx:ch_idx+1]
        val_ch   = val_windows[:, :, ch_idx:ch_idx+1] if len(val_windows) > 0 else None
        m.fit(train_ch, val_ch)
        m.save(save_path)
    lstm_models.append(m)

print(f'\nAll {len(lstm_models)} LSTM models ready')

## Phase 3b — Transformer Autoencoder Training
Same architecture pattern as LSTM but uses sinusoidal positional encoding + self-attention encoder.

In [ ]:
# ── Phase 3b: Train Transformer autoencoder per sEMG channel ────────────────
os.makedirs(os.path.join(MODEL_DIR, 'transformer'), exist_ok=True)

transformer_models = []
for ch_idx, ch_name in enumerate(SEMG_CHANNELS):
    save_path = os.path.join(MODEL_DIR, 'transformer', f'transformer_ch{ch_idx:02d}.pt')
    m = TransformerModel(channel_name=ch_name, window_size=WINDOW_SIZE)
    if os.path.exists(save_path):
        print(f'  [Transformer] ch {ch_idx+1:02d}/{len(SEMG_CHANNELS)}  {ch_name:<30s} — loaded')
        m.load(save_path)
    else:
        print(f'  [Transformer] ch {ch_idx+1:02d}/{len(SEMG_CHANNELS)}  {ch_name:<30s} — training')
        train_ch = train_windows[:, :, ch_idx:ch_idx+1]
        val_ch   = val_windows[:, :, ch_idx:ch_idx+1] if len(val_windows) > 0 else None
        m.fit(train_ch, val_ch)
        m.save(save_path)
    transformer_models.append(m)

print(f'\nAll {len(transformer_models)} Transformer models ready')

## Phase 3c — Semi-Supervised Training
Jointly trains LSTM and Transformer models for reconstruction and classification using synthetically augmented windows. Weights are saved in isolated directories.

In [ ]:
# ── Phase 3c: Semi-Supervised Training ──────────────────────────────────────
active_semisup_models: Dict[str, object] = {}

if USE_SEMI_SUPERVISED:
    for model_key in MODELS:
        model_class = LSTMModel if model_key == 'lstm' else TransformerModel
        out_dir_name = f'{model_key}_semisup'
        os.makedirs(os.path.join(MODEL_DIR, out_dir_name), exist_ok=True)
        
        semisup_models = []
        for ch_idx, ch_name in enumerate(SEMG_CHANNELS):
            save_path = os.path.join(MODEL_DIR, out_dir_name, f'{model_key}_semisup_ch{ch_idx:02d}.pt')
            m = model_class(
                channel_name=ch_name, 
                window_size=WINDOW_SIZE,
                use_classifier=True,
                lambda_cls=LAMBDA_CLS,
                augmentation_fraction=AUGMENTATION_FRACTION
            )
            if os.path.exists(save_path):
                print(f'  [{MODEL_DISPLAY[model_key]} Semi-Sup] ch {ch_idx+1:02d}/{len(SEMG_CHANNELS)}  {ch_name:<30s} — loaded')
                m.load(save_path)
            else:
                print(f'  [{MODEL_DISPLAY[model_key]} Semi-Sup] ch {ch_idx+1:02d}/{len(SEMG_CHANNELS)}  {ch_name:<30s} — training')
                train_ch = train_windows[:, :, ch_idx:ch_idx+1]
                val_ch   = val_windows[:, :, ch_idx:ch_idx+1] if len(val_windows) > 0 else None
                m.fit(train_ch, val_ch)
                m.save(save_path)
            semisup_models.append(m)
        active_semisup_models[model_key] = semisup_models
    print(f'\nAll semi-supervised models ready')
else:
    print('Skipping semi-supervised training (USE_SEMI_SUPERVISED = False)')


## Phase 4 — Anomaly Thresholds
Threshold = μ + 3σ over training reconstruction errors. Training errors are also saved as `.npy` for multi-threshold evaluation.

In [ ]:
# ── Phase 4: Compute per-channel anomaly thresholds from TRAIN errors ────────
active_models: Dict[str, object] = {}
if 'lstm'        in MODELS: active_models['lstm']        = lstm_models
if 'transformer' in MODELS: active_models['transformer'] = transformer_models

thresholds: Dict[str, Dict[str, float]] = {}

for model_key, model_list in active_models.items():
    print(f'\n[Thresholds] {MODEL_DISPLAY[model_key]} …')
    ch_thresh = {}
    for ch_idx, ch_name in enumerate(SEMG_CHANNELS):
        errs = model_list[ch_idx].score(train_windows[:, :, ch_idx:ch_idx+1])
        th   = compute_threshold(errs)           # mu + 3*sigma
        ch_thresh[ch_name] = th
        save_train_errors(errs, ch_name, MODEL_DISPLAY[model_key], SEMG_OUT)
        print(f'  {ch_name:<32s}: {th:.6f}')
    thresholds[model_key] = ch_thresh

print('\nAll thresholds computed and train errors saved')

## Phase 4b — Semi-Supervised Thresholds
Computes the baseline reconstruction thresholds for the semi-supervised models. Cached separately.

In [ ]:
# ── Phase 4b: Compute anomaly thresholds for Semi-Supervised models ──────────
semisup_thresholds: Dict[str, Dict[str, float]] = {}

if USE_SEMI_SUPERVISED:
    for model_key, model_list in active_semisup_models.items():
        print(f'\n[Thresholds] {MODEL_DISPLAY[model_key]} (Semi-Supervised) …')
        ch_thresh = {}
        for ch_idx, ch_name in enumerate(SEMG_CHANNELS):
            errs = model_list[ch_idx].score(train_windows[:, :, ch_idx:ch_idx+1])
            th   = compute_threshold(errs)           # mu + 3*sigma
            ch_thresh[ch_name] = th
            print(f'  {ch_name:<32s}: {th:.6f}')
        semisup_thresholds[model_key] = ch_thresh
    print('\nAll semi-supervised thresholds computed')
else:
    print('Skipping semi-supervised thresholds (USE_SEMI_SUPERVISED = False)')


## Phase 5 — Test Scoring with Synthetic Anomaly Injection
For each test window, three anomaly types × three severities are injected:
- **Amplitude scale** — simulates reduced range of motion
- **Time warp** — simulates asymmetric gait timing
- **Time shift** — simulates delayed muscle activation
- **Combined** — all three at moderate severity

In [ ]:
# ── Phase 5: Score test subjects — clean + synthetic anomalies ───────────────
anomaly_conditions = build_anomaly_conditions(SEVERITIES)
print(f'Anomaly conditions: {len(anomaly_conditions)}')
print(f'  (3 types x {len(SEVERITIES)} severities + 1 combined)')

total = len(TEST_SUBS) * len(MOVEMENTS)
done  = 0

for sub in TEST_SUBS:
    for mov in MOVEMENTS:
        done += 1
        print(f'[{done:3d}/{total}] {sub}/{mov}', end=' … ', flush=True)

        _, windows, meta = process_trial(BASE_DIR, sub, mov, scaling_params)
        if windows is None or len(windows) == 0:
            print('skipped (no windows)')
            continue

        n_win       = len(windows)
        sub_out_dir = os.path.join(SEMG_OUT, sub)
        os.makedirs(sub_out_dir, exist_ok=True)

        # Combine baseline and semi-supervised models for uniform processing
        runs = [(active_models, thresholds, False)]
        if USE_SEMI_SUPERVISED:
            runs.append((active_semisup_models, semisup_thresholds, True))

        for model_group, thresh_group, is_semisup in runs:
            for model_key, model_list in model_group.items():
                model_name = MODEL_DISPLAY[model_key]
                out_model_name = f'{model_name}-SemiSup' if is_semisup else model_name
                all_rows: List[Dict] = []

                # A) Clean windows
                for ch_idx, ch_name in enumerate(SEMG_CHANNELS):
                    th   = thresh_group[model_key][ch_name]
                    ch_win = windows[:, :, ch_idx:ch_idx+1]
                    errs = model_list[ch_idx].score(ch_win)
                    preds = label_windows(errs, th)
                    
                    cls_probs = None
                    if is_semisup:
                        cls_probs = model_list[ch_idx].anomaly_probability(ch_win)
                    
                    all_rows.extend(build_output_rows(
                        meta, errs, preds, ch_name, sub, mov, out_model_name,
                        is_synthetic_anomaly=0, anomaly_type='none',
                        window_id_offset=0, severity=0.0,
                        classifier_probs=cls_probs
                    ))

                # B) Anomalous windows (all conditions)
                for cond_idx, (inj_fn, inj_kw, atype, sev_label) in enumerate(anomaly_conditions):
                    anom_win = np.empty_like(windows)
                    for wi in range(n_win):
                        for ci in range(len(SEMG_CHANNELS)):
                            anom_win[wi, :, ci], _ = inj_fn(windows[wi, :, ci], **inj_kw)

                    w_offset = n_win * (cond_idx + 1)
                    sev_num  = inj_kw.get('severity', DEFAULT_SEVERITIES.get(sev_label, 0.35))

                    for ch_idx, ch_name in enumerate(SEMG_CHANNELS):
                        th   = thresh_group[model_key][ch_name]
                        ch_win = anom_win[:, :, ch_idx:ch_idx+1]
                        errs = model_list[ch_idx].score(ch_win)
                        preds = label_windows(errs, th)
                        
                        cls_probs = None
                        if is_semisup:
                            cls_probs = model_list[ch_idx].anomaly_probability(ch_win)
                        
                        all_rows.extend(build_output_rows(
                            meta, errs, preds, ch_name, sub, mov, out_model_name,
                            is_synthetic_anomaly=1, anomaly_type=atype,
                            window_id_offset=w_offset, severity=float(sev_num),
                            classifier_probs=cls_probs
                        ))

                # Save CSV
                out_path = os.path.join(sub_out_dir, f'{sub}_{mov}_{out_model_name}_scores.csv')
                col_order = [
                    'subject_id', 'modality', 'channel_name', 'movement', 'window_id',
                    'window_start_time', 'window_end_time', 'reconstruction_error',
                    'is_synthetic_anomaly', 'anomaly_type', 'predicted_label',
                    'model_name', 'severity', 'anomaly_probability'
                ]
                out_df = pd.DataFrame(all_rows)
                out_df = out_df[[c for c in col_order if c in out_df.columns]]
                out_df.to_csv(out_path, index=False)

        print('done')

print(f'\nScoring complete — {done} trials processed')


## Phase 6 — Evaluation
Computes Recall, Precision, F1 overall and broken down by severity level. Results saved to `evaluation_summary.csv`.

In [ ]:
# ── Phase 6: Evaluation — Recall, Precision, F1 ─────────────────────────────
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix

results: Dict[str, Dict] = {}

models_to_evaluate = []
for m in MODELS:
    models_to_evaluate.append((MODEL_DISPLAY[m], 'Baseline', False))
    if USE_SEMI_SUPERVISED:
        models_to_evaluate.append((MODEL_DISPLAY[m] + '-SemiSup', 'SemiSup-Recon', False))
        models_to_evaluate.append((MODEL_DISPLAY[m] + '-SemiSup', 'SemiSup-Class', True))

for base_name, variant, use_classifier in models_to_evaluate:
    dfs = []
    for sub in TEST_SUBS:
        sub_dir = os.path.join(SEMG_OUT, sub)
        if not os.path.isdir(sub_dir): continue
        for fname in os.listdir(sub_dir):
            if fname.endswith(f'_{base_name}_scores.csv'):
                dfs.append(pd.read_csv(os.path.join(sub_dir, fname)))

    if not dfs:
        print(f'No CSV results found for {base_name}')
        continue

    df     = pd.concat(dfs, ignore_index=True)
    y_true = df['is_synthetic_anomaly'].values
    
    if use_classifier:
        y_pred = (df['anomaly_probability'].values > 0.5).astype(int)
    else:
        y_pred = df['predicted_label'].values

    rec  = recall_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    display_key = f'{base_name.replace("-SemiSup", "")} ({variant})'
    results[display_key] = {
        'BaseModel': base_name,
        'Variant': variant,
        'Recall': rec, 'Precision': prec, 'F1': f1,
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
        'Total': len(df),
    }

    print(f'\n=== {display_key} ===')
    print(f'  Recall    : {rec:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  F1 Score  : {f1:.4f}')
    print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}  Total={len(df)}')

# Save summary CSV
summary_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'model'})
summary_path = os.path.join(SEMG_OUT, 'evaluation_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f'\nSummary saved -> {summary_path}')
print(summary_df[['model', 'Recall', 'Precision', 'F1']].to_string(index=False))


## Phase 6b — Comprehensive Post-hoc Evaluation Framework
Executes the full post-hoc evaluation framework, generating publication-ready LaTeX/Markdown tables and statistics:
1. **Threshold comparison table**: side-by-side values, relative differences, and window label agreement rate.
2. **Statistical validation**: Kruskal-Wallis H-test on severity strata, Dunn/Mann-Whitney post-hoc tests, and 95% bootstrap CIs.
3. **Non-monotonicity analysis**: quantitative diagnostics (Spearman correlation, saturation ratio) explaining gait anomaly behavior.
4. **Severity profile curves**: error-bar plotting of reconstruction errors across severity levels.

In [ ]:
# Execute evaluation framework with all models config
!python evaluate.py --config configs/all_models.yaml --skip_fusion

## Phase 7 — Visualisation
Bar chart (Recall/Precision/F1) + confusion matrix for the first model. Saved to `results_summary.png`.

In [ ]:
# ── Phase 7: Visualise results ───────────────────────────────────────────────
import matplotlib.pyplot as plt

if not results:
    print('No results to plot — run the evaluation cell first.')
else:
    models  = list(results.keys())
    metrics = ['Recall', 'Precision', 'F1']
    x       = np.arange(len(models))
    width   = 0.25
    colors  = ['#4e9af1', '#f16c4e', '#4ef19a']

    # Adjust figure width dynamically if we have many models/variants
    fig_width = max(14, len(models) * 2)
    fig, axes = plt.subplots(1, 2, figsize=(fig_width, 5))
    fig.suptitle(
        'sEMG Anomaly Detection — Model Comparison\n' 
        f'Movements: {MOVEMENTS}   Severities: {SEVERITIES}',
        fontsize=13, fontweight='bold'
    )

    # ── Bar chart: Recall / Precision / F1 ────────────────────────────────────
    ax = axes[0]
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        vals = [results[m][metric] for m in models]
        bars = ax.bar(x + i * width, vals, width, label=metric,
                      color=color, alpha=0.85, edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.01,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x + width)
    ax.set_xticklabels(models, fontsize=10, rotation=45 if len(models) > 2 else 0, ha='right' if len(models) > 2 else 'center')
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.set_title('Recall / Precision / F1')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(axis='y', alpha=0.3)

    # ── Confusion matrix heatmap for the first available model ────────────────
    ax2  = axes[1]
    m0   = models[0]
    cm   = np.array([[results[m0]['TN'], results[m0]['FP']],
                     [results[m0]['FN'], results[m0]['TP']]])
    im   = ax2.imshow(cm, cmap='Blues')
    lbls = ['Normal', 'Anomaly']
    ax2.set_xticks([0, 1]); ax2.set_xticklabels([f'Pred {l}' for l in lbls])
    ax2.set_yticks([0, 1]); ax2.set_yticklabels([f'True {l}' for l in lbls])
    for r in range(2):
        for c in range(2):
            ax2.text(c, r, f'{cm[r, c]:,}',
                     ha='center', va='center', fontsize=13,
                     color='white' if cm[r, c] > cm.max() / 2 else 'black')
    ax2.set_title(f'Confusion Matrix — {m0}')
    plt.colorbar(im, ax=ax2)

    plt.tight_layout()
    plot_path = os.path.join(SEMG_OUT, 'results_summary.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved -> {plot_path}')
